# Graphes de Patients basés sur les Embeddings ECG

**Candidature Thèse CIFRE - CReSTIC**

---

## Contexte

Ce notebook démontre l'approche proposée dans le sujet de thèse :
- Extraction d'**embeddings** à partir de signaux ECG réels (dataset PTB-XL)
- Construction d'un **graphe de patients** basé sur la similarité
- Identification de **patients prototypiques** et **cas atypiques**
- Application au **raisonnement clinique par analogie**

**Dataset** : PTB-XL (PhysioNet) - 21 799 ECG 12-dérivations annotés

**Modèle** : Auto-encodeur CNN 1D pour compression intelligente des signaux

## 1. Configuration et Imports

## 2. Chargement et Exploration du Dataset PTB-XL

In [ ]:
# Paramètres
MAX_SAMPLES = 1000  # Nombre d'ECG à utiliser (1000 pour test rapide)
SAMPLING_RATE = 100  # Hz
RANDOM_SEED = 42

# Charger le dataset
loader = PTBXLDataLoader(data_dir='./ptb-xl-data', sampling_rate=SAMPLING_RATE)
metadata = loader.load_metadata()

print(f"📊 Dataset PTB-XL chargé")
print(f"   Total ECG disponibles : {len(metadata)}")
print(f"   Patients uniques : {metadata['patient_id'].nunique()}")
print(f"   Échantillon à utiliser : {MAX_SAMPLES}")

In [ ]:
# Statistiques démographiques
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des âges
axes[0].hist(metadata['age'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Âge (années)', fontsize=12)
axes[0].set_ylabel('Nombre de patients', fontsize=12)
axes[0].set_title('Distribution des âges dans PTB-XL', fontsize=14, fontweight='bold')
axes[0].axvline(metadata['age'].median(), color='red', linestyle='--', label=f'Médiane: {metadata["age"].median():.0f} ans')
axes[0].legend()

# Répartition par sexe
sex_counts = metadata['sex'].value_counts()
axes[1].bar(['Hommes', 'Femmes'], [sex_counts.get(0, 0), sex_counts.get(1, 0)], 
            color=['#3498db', '#e74c3c'], edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Nombre de patients', fontsize=12)
axes[1].set_title('Répartition par sexe', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, max(sex_counts) * 1.1)

plt.tight_layout()
plt.savefig('rapport/figures/01_demographics.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure sauvegardée : rapport/figures/01_demographics.png")

## 3. Chargement des Signaux ECG

In [ ]:
# Charger les signaux
print(f"Chargement de {MAX_SAMPLES} signaux ECG...")
signals, valid_metadata = loader.load_all_signals(max_samples=MAX_SAMPLES)

print(f"\n✅ Signaux chargés : {signals.shape}")
print(f"   - Échantillons : {signals.shape[0]}")
print(f"   - Points temporels : {signals.shape[1]} (10 sec à {SAMPLING_RATE} Hz)")
print(f"   - Dérivations : {signals.shape[2]} (I, II, III, AVL, AVR, AVF, V1-V6)")

In [ ]:
# Visualiser un exemple d'ECG
fig, axes = plt.subplots(12, 1, figsize=(15, 12), sharex=True)
lead_names = ['I', 'II', 'III', 'AVL', 'AVR', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

example_ecg = signals[0]  # Premier ECG
time = np.arange(signals.shape[1]) / SAMPLING_RATE

for i, (ax, lead_name) in enumerate(zip(axes, lead_names)):
    ax.plot(time, example_ecg[:, i], linewidth=0.8, color='#2c3e50')
    ax.set_ylabel(lead_name, fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(example_ecg[:, i].min() - 0.1, example_ecg[:, i].max() + 0.1)

axes[-1].set_xlabel('Temps (secondes)', fontsize=12)
fig.suptitle('Exemple d\'ECG 12-dérivations (Patient PTB-XL)', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('rapport/figures/02_example_ecg.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure sauvegardée : rapport/figures/02_example_ecg.png")

## 4. Prétraitement des Signaux

In [ ]:
# Normalisation Z-score
normalized_signals = loader.preprocess_signals(signals)

print(f"✅ Signaux normalisés")
print(f"   Moyenne : {normalized_signals.mean():.6f}")
print(f"   Écart-type : {normalized_signals.std():.6f}")
print(f"   Min/Max : {normalized_signals.min():.3f} / {normalized_signals.max():.3f}")

In [ ]:
# Split train/test
X_train, X_test, meta_train, meta_test = loader.get_train_test_split(
    test_size=0.2,
    random_state=RANDOM_SEED
)

print(f"✅ Split train/test")
print(f"   Train : {X_train.shape[0]} ECG ({X_train.shape[0]/len(normalized_signals)*100:.1f}%)")
print(f"   Test  : {X_test.shape[0]} ECG ({X_test.shape[0]/len(normalized_signals)*100:.1f}%)")

## 5. Construction de l'Auto-encodeur CNN 1D

In [ ]:
# Paramètres du modèle
INPUT_SHAPE = (X_train.shape[1], X_train.shape[2])  # (1000, 12)
EMBEDDING_DIM = 64
FILTERS = [32, 64, 128, 256]
LEARNING_RATE = 0.001
EPOCHS = 50
BATCH_SIZE = 32

print(f"📐 Architecture de l'auto-encodeur")
print(f"   Input shape : {INPUT_SHAPE}")
print(f"   Embedding dimension : {EMBEDDING_DIM}")
print(f"   Filtres convolutifs : {FILTERS}")
print(f"   Taux de compression : {np.prod(INPUT_SHAPE) / EMBEDDING_DIM:.1f}x")

In [ ]:
# Créer et compiler le modèle
ae = ECGAutoencoder(
    input_shape=INPUT_SHAPE,
    embedding_dim=EMBEDDING_DIM,
    filters=FILTERS
)

ae.compile_model(learning_rate=LEARNING_RATE)

print("\n✅ Modèle créé et compilé")
print("\n📋 Résumé de l'encodeur :")
ae.encoder.summary()

## 6. Entraînement de l'Auto-encodeur

In [ ]:
%%time
# Entraîner le modèle
print(f"🚀 Début de l'entraînement ({EPOCHS} époques)...\n")

history = ae.train(
    X_train=X_train,
    X_val=X_test,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

print("\n✅ Entraînement terminé !")

In [ ]:
# Visualiser les courbes d'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Époque', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Évolution de la Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Train', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation', linewidth=2)
axes[1].set_xlabel('Époque', fontsize=12)
axes[1].set_ylabel('MAE', fontsize=12)
axes[1].set_title('Évolution de la MAE', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rapport/figures/03_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure sauvegardée : rapport/figures/03_training_curves.png")
print(f"\n📊 Loss finale : Train = {history.history['loss'][-1]:.6f}, Val = {history.history['val_loss'][-1]:.6f}")

## 7. Extraction des Embeddings

In [ ]:
# Utiliser tout le dataset (train + test) pour le graphe
all_signals = np.concatenate([X_train, X_test], axis=0)

# Extraire les embeddings
embeddings = ae.get_embeddings(all_signals)

print(f"✅ Embeddings extraits : shape {embeddings.shape}")
print(f"   Compression : {np.prod(INPUT_SHAPE)} → {EMBEDDING_DIM} ({np.prod(INPUT_SHAPE)/EMBEDDING_DIM:.1f}x)")
print(f"\n📊 Statistiques des embeddings :")
print(f"   Moyenne : {embeddings.mean():.4f}")
print(f"   Écart-type : {embeddings.std():.4f}")
print(f"   Min/Max : {embeddings.min():.4f} / {embeddings.max():.4f}")

## 8. Construction du Graphe de Patients

In [ ]:
# Construire le graphe
builder = PatientGraphBuilder(
    embeddings=embeddings,
    metadata=valid_metadata
)

# Calculer la similarité
similarity_matrix = builder.compute_similarity_matrix(metric='cosine')

# Construire le graphe K-NN
K_NEIGHBORS = 10
graph = builder.build_knn_graph(k=K_NEIGHBORS)

print(f"\n✅ Graphe construit")
print(f"   Métrique : Similarité cosinus")
print(f"   K voisins : {K_NEIGHBORS}")
print(f"   Nœuds : {graph.number_of_nodes()}")
print(f"   Arêtes : {graph.number_of_edges()}")
print(f"   Degré moyen : {sum(dict(graph.degree()).values()) / graph.number_of_nodes():.2f}")

In [ ]:
# Visualiser la distribution de similarité
plt.figure(figsize=(10, 6))

# Extraire toutes les similarités (triangulaire supérieure)
similarities = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]

plt.hist(similarities, bins=50, edgecolor='black', alpha=0.7, color='#3498db')
plt.xlabel('Similarité cosinus', fontsize=12)
plt.ylabel('Nombre de paires de patients', fontsize=12)
plt.title('Distribution des similarités entre patients', fontsize=14, fontweight='bold')
plt.axvline(similarities.mean(), color='red', linestyle='--', label=f'Moyenne: {similarities.mean():.3f}')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rapport/figures/04_similarity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure sauvegardée : rapport/figures/04_similarity_distribution.png")

## 9. Calcul des Métriques Cognitives

In [ ]:
# Calculer représentativité et atypicité
representativeness = builder.compute_representativeness()
atypicality = builder.compute_atypicality()

print(f"✅ Métriques calculées")
print(f"\n📊 Représentativité :")
print(f"   Moyenne : {representativeness.mean():.4f}")
print(f"   Écart-type : {representativeness.std():.4f}")
print(f"   Min/Max : {representativeness.min():.4f} / {representativeness.max():.4f}")

print(f"\n📊 Atypicité :")
print(f"   Moyenne : {atypicality.mean():.4f}")
print(f"   Écart-type : {atypicality.std():.4f}")
print(f"   Min/Max : {atypicality.min():.4f} / {atypicality.max():.4f}")

In [ ]:
# Visualiser les métriques
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Représentativité
axes[0].hist(representativeness, bins=30, edgecolor='black', alpha=0.7, color='#27ae60')
axes[0].set_xlabel('Score de représentativité', fontsize=12)
axes[0].set_ylabel('Nombre de patients', fontsize=12)
axes[0].set_title('Distribution de la Représentativité', fontsize=14, fontweight='bold')
axes[0].axvline(representativeness.mean(), color='red', linestyle='--', 
                label=f'Moyenne: {representativeness.mean():.3f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Atypicité
axes[1].hist(atypicality, bins=30, edgecolor='black', alpha=0.7, color='#e74c3c')
axes[1].set_xlabel('Score d\'atypicité', fontsize=12)
axes[1].set_ylabel('Nombre de patients', fontsize=12)
axes[1].set_title('Distribution de l\'Atypicité', fontsize=14, fontweight='bold')
axes[1].axvline(atypicality.mean(), color='red', linestyle='--',
                label=f'Moyenne: {atypicality.mean():.3f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rapport/figures/05_metrics_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure sauvegardée : rapport/figures/05_metrics_distribution.png")

## 10. Identification des Prototypes et Cas Atypiques

In [ ]:
# Identifier les patients spéciaux
N_SPECIAL = min(10, len(embeddings) // 10)

prototypes = builder.identify_prototypes(n_prototypes=N_SPECIAL)
atypical_patients = builder.identify_atypical(n_atypical=N_SPECIAL)

print(f"\n✅ Patients identifiés")
print(f"   Prototypes : {len(prototypes)}")
print(f"   Atypiques : {len(atypical_patients)}")

In [ ]:
# Scatter plot représentativité vs atypicité
plt.figure(figsize=(10, 8))

# Tous les patients
plt.scatter(representativeness, atypicality, 
            c='lightgray', alpha=0.5, s=30, label='Patients réguliers')

# Prototypes
plt.scatter(representativeness[prototypes], atypicality[prototypes],
            c='green', s=150, marker='*', edgecolors='black', linewidths=1.5,
            label='Prototypes', zorder=5)

# Atypiques
plt.scatter(representativeness[atypical_patients], atypicality[atypical_patients],
            c='red', s=150, marker='^', edgecolors='black', linewidths=1.5,
            label='Atypiques', zorder=5)

plt.xlabel('Représentativité', fontsize=12)
plt.ylabel('Atypicité', fontsize=12)
plt.title('Carte des Patients : Prototypes vs Atypiques', fontsize=14, fontweight='bold')
plt.legend(fontsize=10, loc='best')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rapport/figures/06_prototypes_vs_atypical.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure sauvegardée : rapport/figures/06_prototypes_vs_atypical.png")

## 11. Sauvegarde des Résultats

In [ ]:
import json

# Sauvegarder le modèle
model_path = 'output/models/ecg_autoencoder'
ae.save(model_path)
print(f"✅ Modèle sauvegardé : {model_path}")

# Sauvegarder les embeddings
np.save('output/embeddings/embeddings.npy', embeddings)
valid_metadata.to_csv('output/embeddings/metadata.csv', index=False)
print(f"✅ Embeddings sauvegardés : output/embeddings/")

# Sauvegarder le graphe
graph_data = builder.export_graph_data()
graph_data['prototypes'] = prototypes
graph_data['atypical'] = atypical_patients

with open('output/graph/graph_data.json', 'w') as f:
    json.dump(graph_data, f, indent=2)

builder.save_graph('output/graph/patient_graph.graphml')
print(f"✅ Graphe sauvegardé : output/graph/")

# Sauvegarder les métriques
metrics = {
    'representativeness': representativeness.tolist(),
    'atypicality': atypicality.tolist(),
    'prototypes': prototypes,
    'atypical': atypical_patients
}

with open('output/graph/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"✅ Métriques sauvegardées : output/graph/metrics.json")

## 12. Génération du Rapport Final

In [ ]:
# Statistiques finales pour le rapport
rapport_stats = {
    'dataset': {
        'n_patients': len(embeddings),
        'sampling_rate': SAMPLING_RATE,
        'signal_duration': 10,
        'n_leads': 12
    },
    'model': {
        'embedding_dim': EMBEDDING_DIM,
        'compression_ratio': float(np.prod(INPUT_SHAPE) / EMBEDDING_DIM),
        'final_train_loss': float(history.history['loss'][-1]),
        'final_val_loss': float(history.history['val_loss'][-1])
    },
    'graph': {
        'n_nodes': graph.number_of_nodes(),
        'n_edges': graph.number_of_edges(),
        'avg_degree': float(sum(dict(graph.degree()).values()) / graph.number_of_nodes()),
        'k_neighbors': K_NEIGHBORS
    },
    'metrics': {
        'avg_representativeness': float(representativeness.mean()),
        'avg_atypicality': float(atypicality.mean()),
        'n_prototypes': len(prototypes),
        'n_atypical': len(atypical_patients)
    }
}

with open('rapport/statistics.json', 'w') as f:
    json.dump(rapport_stats, f, indent=2)

print("✅ Statistiques sauvegardées : rapport/statistics.json")
print("\n" + "="*80)
print("RAPPORT COMPLET GÉNÉRÉ !")
print("="*80)
print("\nFichiers disponibles :")
print("  📊 Figures : rapport/figures/ (6 graphiques)")
print("  📈 Statistiques : rapport/statistics.json")
print("  🧠 Modèle : output/models/")
print("  🔢 Embeddings : output/embeddings/")
print("  🕸️  Graphe : output/graph/")